# Ejercicio 9: Uso de la API de Google Gemini

En este ejercicio vamos a aprender a utilizar la API de OpenAI

## 1. Uso básico

El siguiente código sirve para conectarse con la API de Google Gemini de forma básica

In [1]:
# Instalación de la librería moderna
!pip install -q -U google-genai scikit-learn

from google import genai
from kaggle_secrets import UserSecretsClient

try:
    secret_label = "GEMINI_API_KEY"
    api_key = UserSecretsClient().get_secret(secret_label)
    client = genai.Client(api_key=api_key)
    
    print("Obteniendo la lista de modelos disponibles directamente desde Google...\n")
    modelos_disponibles = []
    
    # 1. Leemos los modelos que tu API Key tiene autorizados
    try:
        for m in client.models.list():
            # Filtramos solo los generativos de texto (gemini) y evitamos los de embedding
            if 'gemini' in m.name.lower() and 'embedding' not in m.name.lower():
                # Limpiamos el prefijo 'models/' si viene incluido
                nombre = m.name.replace('models/', '')
                modelos_disponibles.append(nombre)
    except Exception as e:
        print(f"Error al listar modelos desde el servidor: {e}")
        
    print(f"Modelos detectados en tu cuenta: {modelos_disponibles}\n")
    
    MODELO_ACTIVO = None
    
    # 2. Probamos uno por uno hasta encontrar el que tenga cuota gratuita (Límite > 0)
    for modelo in modelos_disponibles:
        try:
            print(f"Probando cuota de {modelo}...")
            # Enviamos un prompt mínimo para probar la conexión
            response = client.models.generate_content(
                model=modelo, 
                contents="Responde OK"
            )
            MODELO_ACTIVO = modelo
            print(f"¡Éxito! El modelo '{MODELO_ACTIVO}' tiene cuota disponible.\n")
            break
        except Exception as e:
            # Mostramos el error corto para saber por qué falló
            error_msg = str(e).replace('\n', ' ')
            print(f"  -> Descartado: {error_msg[:120]}...")
            
    # 3. Verificamos el resultado de la prueba
    if MODELO_ACTIVO:
        print("-" * 50)
        print(f"Prueba de conocimiento con {MODELO_ACTIVO}:")
        respuesta = client.models.generate_content(
            model=MODELO_ACTIVO, 
            contents="Explica en una oración corta qué es la vectorización de texto en recuperación de información."
        )
        print(respuesta.text)
    else:
        print("\n" + "="*50)
        print("ERROR CRÍTICO DE CUOTA (LÍMITE 0)")
        print("Google AI Studio ha bloqueado el 'Free Tier' para tu API Key.")
        print("Para terminar la tarea necesitas crear un proyecto nuevo en Google Cloud/AI Studio,")
        print("o generar una API Key desde otra cuenta de Google distinta.")
        print("="*50)

except Exception as e:
    print(f"Error general: {e}. Revisa el secreto {secret_label}.")

Obteniendo la lista de modelos disponibles directamente desde Google...

Modelos detectados en tu cuenta: ['gemini-2.5-flash', 'gemini-2.5-pro', 'gemini-2.0-flash', 'gemini-2.0-flash-001', 'gemini-2.0-flash-lite-001', 'gemini-2.0-flash-lite', 'gemini-2.5-flash-preview-tts', 'gemini-2.5-pro-preview-tts', 'gemini-flash-latest', 'gemini-flash-lite-latest', 'gemini-pro-latest', 'gemini-2.5-flash-lite', 'gemini-2.5-flash-image', 'gemini-3-pro-preview', 'gemini-3-flash-preview', 'gemini-3.1-pro-preview', 'gemini-3.1-pro-preview-customtools', 'gemini-3.1-flash-lite-preview', 'gemini-3.1-flash-lite', 'gemini-3-pro-image-preview', 'gemini-3-pro-image', 'gemini-3.1-flash-image-preview', 'gemini-3.1-flash-image', 'gemini-3.1-flash-lite-image', 'gemini-3.5-flash', 'gemini-omni-flash-preview', 'gemini-3.1-flash-tts-preview', 'gemini-robotics-er-1.5-preview', 'gemini-robotics-er-1.6-preview', 'gemini-2.5-computer-use-preview-10-2025', 'gemini-2.5-flash-native-audio-latest', 'gemini-2.5-flash-native-

## 2. Retrieval
Vamos a construir un sistema de Recuperación Aumentada por Generación (RAG) convirtiendo textos en vectores semánticos (embeddings).

### 2.1 Cargo el corpus de 20 News Groups
Para este ejercicio, descargaremos una muestra de foros sobre motocicletas y componentes de hardware para evaluar la búsqueda sobre lenguaje técnico.

In [2]:
from sklearn.datasets import fetch_20newsgroups

# Descargamos los datos eliminando metadatos para obtener solo el cuerpo del mensaje
categorias = ['rec.motorcycles', 'comp.sys.ibm.pc.hardware']
print("Descargando corpus de 20 News Groups...")
data = fetch_20newsgroups(subset='train', categories=categorias, remove=('headers', 'footers', 'quotes'))

# REDUCCIÓN ANTI-BLOQUEO: Tomamos solo 10 documentos para asegurar que la API 
# gratuita procese todo a la perfección sin llegar al límite de peticiones (429).
documentos = [doc for doc in data.data if len(doc.split()) > 20][:10]

print(f"Dataset cargado. Documentos listos para vectorizar: {len(documentos)}")

Descargando corpus de 20 News Groups...
Dataset cargado. Documentos listos para vectorizar: 10


### 2.2 Transformo a embeddings
Cada documento será mapeado a un espacio vectorial utilizando el modelo de embeddings de texto de Gemini.

In [4]:
import numpy as np
import time

print("Buscando el modelo de vectorización (embeddings) habilitado en tu cuenta...\n")
modelo_embedding_activo = None

# 1. Buscamos el nombre exacto del modelo en los servidores de Google
try:
    for m in client.models.list():
        # Filtramos buscando la palabra 'embed' o 'embedding'
        if 'embed' in m.name.lower():
            modelo_embedding_activo = m.name.replace('models/', '')
            break
except Exception as e:
    print(f"Error consultando modelos: {e}")

# Si por alguna razón la lista falla, aplicamos un respaldo a la versión anterior
if not modelo_embedding_activo:
    modelo_embedding_activo = "embedding-001"

print(f"¡Éxito! Utilizaremos el modelo: {modelo_embedding_activo}")
print("Generando embeddings con pausas de seguridad de 4.5 segundos...\n")

doc_embeddings = []

# 2. Vectorizamos usando el modelo dinámico
for i, doc in enumerate(documentos):
    try:
        result = client.models.embed_content(
            model=modelo_embedding_activo,
            contents=doc,
        )
        doc_embeddings.append(result.embeddings[0].values)
        print(f"Documento {i+1}/{len(documentos)} procesado.")
        time.sleep(4.5) # Pausa obligatoria
    except Exception as e:
        print(f"Error en documento {i+1}: {e}")
        doc_embeddings.append([0.0] * 768)

matriz_embeddings = np.array(doc_embeddings)
print(f"\n¡Transformación completada! Dimensiones de la matriz: {matriz_embeddings.shape}")

Buscando el modelo de vectorización (embeddings) habilitado en tu cuenta...

¡Éxito! Utilizaremos el modelo: gemini-embedding-001
Generando embeddings con pausas de seguridad de 4.5 segundos...

Documento 1/10 procesado.
Documento 2/10 procesado.
Documento 3/10 procesado.
Documento 4/10 procesado.
Documento 5/10 procesado.
Documento 6/10 procesado.
Documento 7/10 procesado.
Documento 8/10 procesado.
Documento 9/10 procesado.
Documento 10/10 procesado.

¡Transformación completada! Dimensiones de la matriz: (10, 3072)


### 2.3 Creo una query y hago la búsqueda
Convertiremos nuestra consulta de texto al mismo espacio vectorial para calcular la distancia geométrica (Similitud del Coseno) contra todos los documentos.

In [5]:
from sklearn.metrics.pairwise import cosine_similarity

# Definimos nuestra consulta técnica sobre mecánica de motores
query = "Instrucciones de revisión mecánica, diagnóstico de embrague y mantenimiento del carburador para un motor de 200cc enfriado por aire."
print(f"Nuestra consulta de búsqueda es: '{query}'\n")

# Vectorizamos la consulta usando EL MISMO modelo dinámico que la Celda 7
query_res = client.models.embed_content(
    model=modelo_embedding_activo,
    contents=query
)

# Ajustamos la forma del arreglo para scikit-learn
query_vector = np.array(query_res.embeddings[0].values).reshape(1, -1)

# Calculamos la similitud del coseno
similitudes = cosine_similarity(query_vector, matriz_embeddings)[0]
print("Cálculo de distancias completado.")

Nuestra consulta de búsqueda es: 'Instrucciones de revisión mecánica, diagnóstico de embrague y mantenimiento del carburador para un motor de 200cc enfriado por aire.'

Cálculo de distancias completado.


## Obtengo los 5 documentos más similares a mi query

In [6]:
# Ordenamos los índices de mayor a menor similitud y tomamos los 5 primeros
top_5_indices = np.argsort(similitudes)[::-1][:5]

print("--- RESULTADOS DE RECUPERACIÓN (TOP 5) ---\n")
for ranking, indice in enumerate(top_5_indices, 1):
    print(f"[{ranking}] Score de Similitud: {similitudes[indice]:.4f}")
    # Mostramos un fragmento de 250 caracteres del documento recuperado
    print(f"Texto recuperado:\n{documentos[indice][:250]}...\n")
    print("-" * 70)

--- RESULTADOS DE RECUPERACIÓN (TOP 5) ---

[1] Score de Similitud: 0.5417
Texto recuperado:


In New Orleans, LA, there was a company making motorcycles for WHEELCHAIR 
bound people!  The rig consists of a flat-bed sidecar rig that the 
wheelchair can be clamped to.  The car has a set of hand controls mounted on 
conventional handlebars!  L...

----------------------------------------------------------------------
[2] Score de Similitud: 0.5237
Texto recuperado:
CB>From: behanna@syl.nj.nec.com (Chris BeHanna)

CB>>|>
CB>>|>  Grf. Dropped my Shoei RF-200 off the seat of my bike while trying to
CB>>|> rock
CB>>|> it onto it's centerstand, chipped the heck out of the paint on it...

CB>        Do I have to be t...

----------------------------------------------------------------------
[3] Score de Similitud: 0.5179
Texto recuperado:

: {> 
: {> 	Howdy,
: {> 
: {> 	The other day I was using Norton's SpeedDisk to optimize my Seagate(125MB) h
: {> problem persisted.  I backed up all essenti

In [8]:
# Paso final RAG: Alimentamos a Gemini con los documentos recuperados para que genere una respuesta experta

contexto_recuperado = "\n\n---\n\n".join([documentos[idx] for idx in top_5_indices])

prompt_final = f"""
Actúa como un experto en mecánica. 
Basándote ÚNICAMENTE en el siguiente contexto recuperado de foros, responde a la pregunta del usuario de forma estructurada. Si el contexto no tiene información exacta, deduce recomendaciones generales mecánicas alineadas al texto.

Pregunta del usuario: {query}

Contexto recuperado:
{contexto_recuperado}
"""

print(f"Generando síntesis final basada en los documentos recuperados usando {MODELO_ACTIVO}...\n")

# Utilizamos la variable dinámica que encontramos en la Celda 2
respuesta_rag = client.models.generate_content(
    model=MODELO_ACTIVO,
    contents=prompt_final
)

print("--- RESPUESTA GENERADA (SISTEMA RAG) ---\n")
print(respuesta_rag.text)

Generando síntesis final basada en los documentos recuperados usando gemini-flash-latest...

--- RESPUESTA GENERADA (SISTEMA RAG) ---

Como experto en mecánica, y basándome en las experiencias, advertencias de seguridad y metodologías de resolución de problemas presentes en el foro provisto, he estructurado las siguientes recomendaciones de revisión, diagnóstico y mantenimiento para un motor de 200cc enfriado por aire.

Dado que el texto no contiene especificaciones técnicas directas de este motor, he **deducido y alineado las mejores prácticas mecánicas** utilizando las analogías y lecciones de seguridad del contexto:

---

### 1. Instrucciones de Revisión Mecánica General (Estabilidad y Seguridad)
*Basado en los incidentes de caída de componentes del foro (caso del casco en el asiento) y las adaptaciones de control:*

*   **Estabilización del vehículo:** Antes de iniciar cualquier inspección en el motor de 200cc, asegúrate de colocar la motocicleta sobre su soporte central (*centerst